# 05 · Tableau Rich Dataset Preparation

## Objective

This notebook creates a **single Tableau-ready dataset** for the Vanguard A/B Testing project.

The goal is to preserve the storytelling thread from the previous phases:

1. **Data Cleaning** → cleaned and structured raw data  
2. **Performance Metrics** → completion, errors and time metrics  
3. **Hypothesis Testing** → statistical validation of the experiment  
4. **Experiment Evaluation** → business interpretation  
5. **Tableau Visualizations** → executive dashboard and story

Instead of using multiple CSV files in Tableau, this notebook creates one enriched CSV at **visit level**. Each row represents one unique client visit and includes:

- Experiment group: Control or Test
- Completion indicators
- Error indicators
- Process efficiency metrics
- Funnel step indicators
- Client demographics and segmentation variables
- Client account and engagement metrics

## Why visit-level?

For the dashboard, the most important unit of analysis is the **visit/session**, not every individual click event.

A visit-level dataset allows Tableau to calculate clean KPIs such as:

- Completion Rate = average of `completion_flag`
- Error Rate = average of `error_flag`
- Average Time = average of `total_minutes`
- Funnel counts = sum of each process step flag

## Output

This notebook exports:

`vanguard_tableau_dashboard.csv`

This file should be used as the **main and only data source** for the Tableau dashboard.


## 1. Imports and configuration

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Input and output files
INPUT_FILE = Path("vanguard_cleaned_merged.csv")
OUTPUT_FILE = Path("vanguard_tableau_rich_dashboard.csv")

## 2. Load cleaned merged dataset

This notebook starts from the cleaned merged dataset created in the Data Cleaning phase.

The code below also standardizes column names to make the rest of the notebook easier to run, even if Tableau or Excel display the fields with spaces/capital letters.

In [2]:
df = pd.read_csv(INPUT_FILE)

print("Original shape:", df.shape)
df.head()

Original shape: (317235, 19)


,client_id,visitor_id,visit_id,process_step,date_time,Variation,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth,age_group,tenure_group,balance_segment,date,year_month
0,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:27:07,Test,5.0,64.0,79.0,Unknown,2.0,189023.86,1.0,4.0,70+,New,Very High,2017-04-17,2017-04
1,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:26:51,Test,5.0,64.0,79.0,Unknown,2.0,189023.86,1.0,4.0,70+,New,Very High,2017-04-17,2017-04
2,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:19:22,Test,5.0,64.0,79.0,Unknown,2.0,189023.86,1.0,4.0,70+,New,Very High,2017-04-17,2017-04
3,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:19:13,Test,5.0,64.0,79.0,Unknown,2.0,189023.86,1.0,4.0,70+,New,Very High,2017-04-17,2017-04
4,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:18:04,Test,5.0,64.0,79.0,Unknown,2.0,189023.86,1.0,4.0,70+,New,Very High,2017-04-17,2017-04


In [3]:
# Standardize column names
# Example: "Client Id" -> "client_id", "Process Step" -> "process_step"

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

print(df.columns.tolist())

['client_id', 'visitor_id', 'visit_id', 'process_step', 'date_time', 'variation', 'clnt_tenure_yr', 'clnt_tenure_mnth', 'clnt_age', 'gendr', 'num_accts', 'bal', 'calls_6_mnth', 'logons_6_mnth', 'age_group', 'tenure_group', 'balance_segment', 'date', 'year_month']


## 3. Validate key columns

The dashboard needs a few essential fields to build the experiment metrics.

In [4]:
required_cols = [
    "client_id",
    "visit_id",
    "variation",
    "process_step",
    "date_time"
]

missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")
else:
    print("All required columns are available.")

All required columns are available.


## 4. Prepare process steps and timestamps

The digital process has a clear order:

1. start  
2. step_1  
3. step_2  
4. step_3  
5. confirm

This order allows us to calculate completion, funnel progression and backward movements.

In [5]:
STEP_ORDER = {
    "start": 0,
    "step_1": 1,
    "step_2": 2,
    "step_3": 3,
    "confirm": 4
}

# Clean required fields
df = df.dropna(subset=["client_id", "visit_id", "variation", "process_step", "date_time"]).copy()

# Convert timestamp
df["date_time"] = pd.to_datetime(df["date_time"], errors="coerce")
df = df.dropna(subset=["date_time"]).copy()

# Map process step order
df["step_index"] = df["process_step"].map(STEP_ORDER)
df = df.dropna(subset=["step_index"]).copy()
df["step_index"] = df["step_index"].astype(int)

# Sort events within each visit
df = df.sort_values(["client_id", "visit_id", "date_time", "step_index"]).reset_index(drop=True)

print("Cleaned event-level shape:", df.shape)
df[["client_id", "visit_id", "variation", "process_step", "date_time", "step_index"]].head(10)

Cleaned event-level shape: (317235, 20)


,client_id,visit_id,variation,process_step,date_time,step_index
0,555,637149525_38041617439_716659,Test,start,2017-04-15 12:57:56,0
1,555,637149525_38041617439_716659,Test,step_1,2017-04-15 12:58:03,1
2,555,637149525_38041617439_716659,Test,step_2,2017-04-15 12:58:35,2
3,555,637149525_38041617439_716659,Test,step_3,2017-04-15 13:00:14,3
4,555,637149525_38041617439_716659,Test,confirm,2017-04-15 13:00:34,4
5,647,40369564_40101682850_311847,Test,start,2017-04-12 15:41:28,0
6,647,40369564_40101682850_311847,Test,step_1,2017-04-12 15:41:35,1
7,647,40369564_40101682850_311847,Test,step_2,2017-04-12 15:41:53,2
8,647,40369564_40101682850_311847,Test,step_3,2017-04-12 15:45:02,3
9,647,40369564_40101682850_311847,Test,confirm,2017-04-12 15:47:45,4


## 5. Create visit-level experiment metrics

Each row in the final Tableau dataset will represent one visit.

Metrics created here:

- `completion_flag`: 1 if the visit reached `confirm`, otherwise 0
- `total_minutes`: time between first and last event in the visit
- `max_step`: furthest step reached in the journey
- `total_events`: number of events recorded during the visit

In [6]:
visit_df = df.groupby("visit_id").agg(
    client_id=("client_id", "first"),
    variation=("variation", "first"),
    reached_confirm=("process_step", lambda x: "confirm" in set(x)),
    max_step=("step_index", "max"),
    start_time=("date_time", "min"),
    end_time=("date_time", "max"),
    total_events=("process_step", "size")
).reset_index()

visit_df["total_minutes"] = (
    visit_df["end_time"] - visit_df["start_time"]
).dt.total_seconds() / 60

visit_df["completion_flag"] = visit_df["reached_confirm"].astype(int)

visit_df.head()

,visit_id,client_id,variation,reached_confirm,max_step,start_time,end_time,total_events,total_minutes,completion_flag
0,100012776_37918976071_457913,3561384,Test,True,4,2017-04-26 13:22:17,2017-04-26 13:23:09,2,0.866667,1
1,100019538_17884295066_43909,7338123,Test,True,4,2017-04-09 16:20:56,2017-04-09 16:24:58,11,4.033333,1
2,100022086_87870757897_149620,2478628,Test,True,4,2017-05-23 20:44:01,2017-05-23 20:47:01,5,3.000000,1
3,100030127_47967100085_936361,105007,Control,False,0,2017-03-22 11:07:49,2017-03-22 11:07:49,1,0.000000,0
4,100037962_47432393712_705583,5623007,Control,False,1,2017-04-14 16:41:51,2017-04-14 16:44:03,4,2.200000,0


## 6. Create error metrics

For this project, an error is defined as a **backward movement** in the process.

Example:

`start → step_1 → step_2 → step_1`

This suggests that the user had to go back, correct something, or repeat part of the journey. It is not necessarily a technical system error; it is a navigation friction indicator.

In [7]:
df["previous_step_index"] = df.groupby("visit_id")["step_index"].shift(1)
df["is_backward"] = df["step_index"] < df["previous_step_index"]

visit_errors = df.groupby("visit_id").agg(
    backward_moves=("is_backward", "sum")
).reset_index()

visit_errors["has_error"] = visit_errors["backward_moves"] > 0
visit_errors["error_flag"] = visit_errors["has_error"].astype(int)

visit_df = visit_df.merge(visit_errors, on="visit_id", how="left")

visit_df[["visit_id", "backward_moves", "has_error", "error_flag"]].head()

,visit_id,backward_moves,has_error,error_flag
0,100012776_37918976071_457913,0,False,0
1,100019538_17884295066_43909,2,True,1
2,100022086_87870757897_149620,0,False,0
3,100030127_47967100085_936361,0,False,0
4,100037962_47432393712_705583,1,True,1


## 7. Create funnel step indicators

Instead of using event-level rows in Tableau, we create one flag per process step.

This keeps the dataset at visit level while still allowing a funnel visualization in Tableau.

For example:

- `reached_start`
- `reached_step_1`
- `reached_step_2`
- `reached_step_3`
- `reached_confirm`

In Tableau, the funnel can be built using the **SUM** of these fields by `variation`.

In [8]:
# Create one row per visit with True/False for each process step
step_flags = pd.crosstab(df["visit_id"], df["process_step"])

# Ensure all expected step columns exist
for step in STEP_ORDER.keys():
    if step not in step_flags.columns:
        step_flags[step] = 0

# Convert counts into binary flags and rename columns
step_flags = (step_flags[list(STEP_ORDER.keys())] > 0).astype(int)
step_flags = step_flags.rename(columns={
    "start": "reached_start",
    "step_1": "reached_step_1",
    "step_2": "reached_step_2",
    "step_3": "reached_step_3",
    "confirm": "reached_confirm_step"
}).reset_index()

visit_df = visit_df.merge(step_flags, on="visit_id", how="left")

# Fill missing step flags with 0
step_flag_cols = [
    "reached_start",
    "reached_step_1",
    "reached_step_2",
    "reached_step_3",
    "reached_confirm_step"
]

visit_df[step_flag_cols] = visit_df[step_flag_cols].fillna(0).astype(int)

visit_df[step_flag_cols].head()

,reached_start,reached_step_1,reached_step_2,reached_step_3,reached_confirm_step
0,0,0,0,0,1
1,1,1,1,1,1
2,1,1,1,1,1
3,1,0,0,0,0
4,1,1,0,0,0


## 8. Add client profile and segmentation fields

This section enriches the Tableau dataset with client-level context.

These fields support demographic and behavioral exploration in Tableau:

- Age
- Age Group
- Gender
- Tenure Group
- Balance Segment
- Number of accounts
- Calls in last 6 months
- Logons in last 6 months

In [9]:
# Keep the first available value per client for demographic/account fields
possible_demo_cols = [
    "client_id",
    "clnt_age",
    "gendr",
    "clnt_tenure_yr",
    "clnt_tenure_mnth",
    "bal",
    "num_accts",
    "calls_6_mnth",
    "logons_6_mnth",
    "age_group",
    "tenure_group",
    "balance_segment",
    "year_month"
]

available_demo_cols = [col for col in possible_demo_cols if col in df.columns]

client_demo = (
    df[available_demo_cols]
    .drop_duplicates(subset=["client_id"])
    .copy()
)

print("Available demographic/profile columns:")
print(available_demo_cols)

client_demo.head()

Available demographic/profile columns:
['client_id', 'clnt_age', 'gendr', 'clnt_tenure_yr', 'clnt_tenure_mnth', 'bal', 'num_accts', 'calls_6_mnth', 'logons_6_mnth', 'age_group', 'tenure_group', 'balance_segment', 'year_month']


,client_id,clnt_age,gendr,clnt_tenure_yr,clnt_tenure_mnth,bal,num_accts,calls_6_mnth,logons_6_mnth,age_group,tenure_group,balance_segment,year_month
0,555,29.5,Unknown,3.0,46.0,25454.66,2.0,2.0,6.0,Under 30,New,Low,2017-04
5,647,57.5,M,12.0,151.0,30525.80,2.0,0.0,4.0,50-70,Loyal,Low,2017-04
10,934,51.0,F,9.0,109.0,32522.88,2.0,0.0,3.0,50-70,Established,Low,2017-04
14,1028,36.0,M,12.0,145.0,103520.22,3.0,1.0,4.0,30-50,Loyal,High,2017-04
23,1104,48.0,Unknown,5.0,66.0,154643.94,3.0,6.0,9.0,30-50,New,Very High,2017-06


In [10]:
# If Age Group does not exist, create it from clnt_age
if "age_group" not in client_demo.columns and "clnt_age" in client_demo.columns:
    age_bins = [0, 29, 39, 49, 59, 69, 200]
    age_labels = ["Under 30", "30-39", "40-49", "50-59", "60-69", "70+"]
    client_demo["age_group"] = pd.cut(
        client_demo["clnt_age"],
        bins=age_bins,
        labels=age_labels,
        include_lowest=True
    ).astype("object")

# If Tenure Group does not exist, create it from clnt_tenure_yr
if "tenure_group" not in client_demo.columns and "clnt_tenure_yr" in client_demo.columns:
    tenure_bins = [-1, 5, 10, 15, 20, 100]
    tenure_labels = ["0-5 years", "6-10 years", "11-15 years", "16-20 years", "20+ years"]
    client_demo["tenure_group"] = pd.cut(
        client_demo["clnt_tenure_yr"],
        bins=tenure_bins,
        labels=tenure_labels
    ).astype("object")

# If Balance Segment does not exist, create it from bal using quartiles
if "balance_segment" not in client_demo.columns and "bal" in client_demo.columns:
    try:
        client_demo["balance_segment"] = pd.qcut(
            client_demo["bal"],
            q=4,
            labels=["Low Balance", "Mid-Low Balance", "Mid-High Balance", "High Balance"],
            duplicates="drop"
        ).astype("object")
    except ValueError:
        client_demo["balance_segment"] = "Not Available"

client_demo.head()

,client_id,clnt_age,gendr,clnt_tenure_yr,clnt_tenure_mnth,bal,num_accts,calls_6_mnth,logons_6_mnth,age_group,tenure_group,balance_segment,year_month
0,555,29.5,Unknown,3.0,46.0,25454.66,2.0,2.0,6.0,Under 30,New,Low,2017-04
5,647,57.5,M,12.0,151.0,30525.80,2.0,0.0,4.0,50-70,Loyal,Low,2017-04
10,934,51.0,F,9.0,109.0,32522.88,2.0,0.0,3.0,50-70,Established,Low,2017-04
14,1028,36.0,M,12.0,145.0,103520.22,3.0,1.0,4.0,30-50,Loyal,High,2017-04
23,1104,48.0,Unknown,5.0,66.0,154643.94,3.0,6.0,9.0,30-50,New,Very High,2017-06


## 9. Merge visit metrics with client profile

In [11]:
tableau_df = visit_df.merge(client_demo, on="client_id", how="left")

# Create clean categorical labels for Tableau
tableau_df["completion_status"] = np.where(
    tableau_df["completion_flag"] == 1,
    "Completed",
    "Not Completed"
)

tableau_df["error_status"] = np.where(
    tableau_df["error_flag"] == 1,
    "Had Backward Movement",
    "No Backward Movement"
)

# Create a clean label for the furthest step reached
max_step_labels = {v: k for k, v in STEP_ORDER.items()}
tableau_df["furthest_step_reached"] = tableau_df["max_step"].map(max_step_labels)

print("Tableau dataset shape:", tableau_df.shape)
tableau_df.head()

Tableau dataset shape: (69205, 33)


,visit_id,client_id,variation,reached_confirm,max_step,start_time,end_time,total_events,total_minutes,completion_flag,backward_moves,has_error,error_flag,reached_start,reached_step_1,reached_step_2,reached_step_3,reached_confirm_step,clnt_age,gendr,clnt_tenure_yr,clnt_tenure_mnth,bal,num_accts,calls_6_mnth,logons_6_mnth,age_group,tenure_group,balance_segment,year_month,completion_status,error_status,furthest_step_reached
0,100012776_37918976071_457913,3561384,Test,True,4,2017-04-26 13:22:17,2017-04-26 13:23:09,2,0.866667,1,0,False,0,0,0,0,0,1,59.5,Unknown,4.0,56.0,63130.44,2.0,6.0,9.0,50-70,New,Medium,2017-04,Completed,No Backward Movement,confirm
1,100019538_17884295066_43909,7338123,Test,True,4,2017-04-09 16:20:56,2017-04-09 16:24:58,11,4.033333,1,2,True,1,1,1,1,1,1,23.5,M,7.0,88.0,26436.73,2.0,6.0,9.0,Under 30,Established,Low,2017-04,Completed,Had Backward Movement,confirm
2,100022086_87870757897_149620,2478628,Test,True,4,2017-05-23 20:44:01,2017-05-23 20:47:01,5,3.000000,1,0,False,0,1,1,1,1,1,47.0,F,16.0,198.0,32456.28,2.0,2.0,5.0,30-50,Loyal,Low,2017-05,Completed,No Backward Movement,confirm
3,100030127_47967100085_936361,105007,Control,False,0,2017-03-22 11:07:49,2017-03-22 11:07:49,1,0.000000,0,0,False,0,1,0,0,0,0,35.0,F,9.0,118.0,34897.47,2.0,3.0,6.0,30-50,Established,Low,2017-03,Not Completed,No Backward Movement,start
4,100037962_47432393712_705583,5623007,Control,False,1,2017-04-14 16:41:51,2017-04-14 16:44:03,4,2.200000,0,1,True,1,1,1,0,0,0,78.0,M,16.0,202.0,146827.14,2.0,5.0,8.0,70+,Loyal,High,2017-04,Not Completed,Had Backward Movement,step_1


## 10. Quality checks

Before exporting, we validate that the numbers match the story we want to tell in Tableau.

In [12]:
kpi_summary = tableau_df.groupby("variation").agg(
    visits=("visit_id", "nunique"),
    clients=("client_id", "nunique"),
    completion_rate=("completion_flag", "mean"),
    error_rate=("error_flag", "mean"),
    avg_total_minutes=("total_minutes", "mean"),
    median_total_minutes=("total_minutes", "median"),
    avg_backward_moves=("backward_moves", "mean")
).reset_index()

kpi_summary

,variation,visits,clients,completion_rate,error_rate,avg_total_minutes,median_total_minutes,avg_backward_moves
0,Control,32128,23439,0.497977,0.202845,4.723381,2.666667,0.299085
1,Test,37077,26873,0.584756,0.270518,5.295703,2.800000,0.438951


In [13]:
# Funnel validation by experiment group
funnel_summary = tableau_df.groupby("variation")[step_flag_cols].sum().reset_index()
funnel_summary

,variation,reached_start,reached_step_1,reached_step_2,reached_step_3,reached_confirm_step
0,Control,30851,23496,20092,18255,15999
1,Test,33100,28228,24449,22136,21681


In [14]:
# Missing values check for key dashboard fields
key_dashboard_cols = [
    "visit_id",
    "client_id",
    "variation",
    "completion_flag",
    "error_flag",
    "total_minutes",
    "age_group",
    "gendr",
    "tenure_group",
    "balance_segment"
]

available_key_cols = [col for col in key_dashboard_cols if col in tableau_df.columns]

missing_summary = tableau_df[available_key_cols].isna().sum().reset_index()
missing_summary.columns = ["column", "missing_values"]
missing_summary

,column,missing_values
0,visit_id,0
1,client_id,0
2,variation,0
3,completion_flag,0
4,error_flag,0
5,total_minutes,0
6,age_group,22
7,gendr,22
8,tenure_group,22
9,balance_segment,22


## 11. Export final Tableau dataset

This is the main CSV to connect in Tableau.

Use this file instead of connecting multiple CSVs separately.

In [15]:
tableau_df.to_csv(OUTPUT_FILE, index=False)

print(f"CSV created successfully: {OUTPUT_FILE}")
print("Final shape:", tableau_df.shape)
print("Columns exported:")
print(tableau_df.columns.tolist())

CSV created successfully: vanguard_tableau_rich_dashboard.csv
Final shape: (69205, 33)
Columns exported:
['visit_id', 'client_id', 'variation', 'reached_confirm', 'max_step', 'start_time', 'end_time', 'total_events', 'total_minutes', 'completion_flag', 'backward_moves', 'has_error', 'error_flag', 'reached_start', 'reached_step_1', 'reached_step_2', 'reached_step_3', 'reached_confirm_step', 'clnt_age', 'gendr', 'clnt_tenure_yr', 'clnt_tenure_mnth', 'bal', 'num_accts', 'calls_6_mnth', 'logons_6_mnth', 'age_group', 'tenure_group', 'balance_segment', 'year_month', 'completion_status', 'error_status', 'furthest_step_reached']


## 12. Suggested Tableau visuals

Use `vanguard_tableau_rich_dashboard.csv` as the only data source.

### Executive KPI charts

1. **Completion Rate by Experiment Group**  
   - Columns: `variation`  
   - Rows: `AVG(completion_flag)`  
   - Format as percentage

2. **Error Rate by Experiment Group**  
   - Columns: `variation`  
   - Rows: `AVG(error_flag)`  
   - Format as percentage

3. **Process Efficiency**  
   - Columns: `variation`  
   - Rows: `AVG(total_minutes)`

### Segment analysis

4. **Completion Rate by Age Group**  
   - Columns: `age_group`  
   - Rows: `AVG(completion_flag)`  
   - Color: `variation`

5. **Completion Rate by Gender**  
   - Columns: `gendr`  
   - Rows: `AVG(completion_flag)`  
   - Color: `variation`

### Funnel analysis

6. **Customer Journey Funnel**  
   Use the following fields as measures:

   - `SUM(reached_start)`
   - `SUM(reached_step_1)`
   - `SUM(reached_step_2)`
   - `SUM(reached_step_3)`
   - `SUM(reached_confirm_step)`

   Then compare by `variation`.

## Storytelling thread

The dashboard should answer one main business question:

**Should Vanguard roll out the Test digital experience to all clients?**

The current story should be framed as:

**The Test experience increased completion rates, but it also introduced more navigation friction and slightly longer completion times. Vanguard should consider rolling it out only after investigating the steps where users experience more friction.**